# Report 2: From the solar wind to the polar ionosphere

A main goal of this report is to explain how changing solar wind and IMF conditions could produce the magnetic variations observed at Qaanaaq/Thule (THL) in Greenland. Section 3 is the main task, while sections 1 and 2 provide background for your event analysis.

Use the code below for downloading data, coordinate conversions, baseline subtraction and model plots. The emphasis is on interpreting observations and models, not programming.

Submit no more than 6 pages, including figures, with roughly two-thirds devoted to section 3. Give units, UTC dates, captions and data sources. Give the information needed to reproduce your results, including model inputs, baseline and averaging intervals. Only include figures that are needed for your explanation.

Data: THL one-minute INTERMAGNET observations through [VirES](https://notebooks.vires.services/notebooks/04c2_geomag-ground-data-vires), one-minute solar wind/IMF and AE from [NASA OMNI](https://omniweb.gsfc.nasa.gov/html/omni_min_data.html), and hourly Dst. Acknowledge the observatory and data providers. VirES needs a configured account. All times are UTC. OMNI solar wind data are already time-shifted from upstream spacecraft to the bow shock. Missing values remain gaps.

In [ ]:
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from viresclient import SwarmRequest
from dipole import Dipole
from apexpy import Apex
from ppigrf.ppigrf import geoc2geod

plt.rcParams.update({"font.size": 11, "axes.grid": True, "font.sans-serif": ["DejaVu Sans"]})
# Keep plots portable without a separate LaTeX installation.
plt.rcParams["text.usetex"] = False

## 1. Explore two months of data

### Download the longer record

The initial interval is May–June 2009, a relatively quiet interval near solar minimum. Its hourly IMF By and Bz means stay within about −10 to +10 nT, and hourly Dst reaches a minimum of −28 nT. Change `start` and `stop` to study other months. Downloads are cached beside the notebook in monthly chunks, so rerunning the notebook reuses the files. The first run can take a bit of time.

The baseline below is a **single constant vector**, the component-wise median over the chosen reference interval. It defines a reference level, not an absolute external magnetic field or a measured quiet-day curve. It preserves changes in daily means. The example reference interval is a starting choice: inspect it before describing it as quiet. Do not subtract each day's mean if you want to study changes between days.

In [ ]:
start          = pd.Timestamp("2009-05-01T00:00:00Z")
stop           = pd.Timestamp("2009-07-01T00:00:00Z")
baseline_start = pd.Timestamp("2009-05-01T00:00:00Z")
baseline_stop  = pd.Timestamp("2009-05-02T00:00:00Z")
use_cache      = True

if not start < stop or not start <= baseline_start < baseline_stop <= stop:
    raise ValueError("Choose start < stop and a baseline interval inside the record.")

cache_directory = Path("report_2_data")
cache_directory.mkdir(exist_ok=True)
chunk_edges = pd.DatetimeIndex([start, *pd.date_range(start, stop, freq="MS"), stop]).unique().sort_values()

In [ ]:
def cached_interval(prefix, suffix, start, stop):
    """Find a cached download that contains the requested interval."""
    for path in sorted(cache_directory.glob(prefix + "_*_" + suffix)):
        dates = path.name[len(prefix) + 1:-(len(suffix) + 1)].split("_")
        if len(dates) != 2:
            continue
        left, right = [pd.to_datetime(date, format="%Y%m%dT%H%M%S", utc=True) for date in dates]
        if left <= start and stop <= right:
            return path
    return None

def download_omni(dataset, columns, start, stop, use_cache=True):
    """Return a UTC-indexed DataFrame for start <= time < stop."""
    query = urlencode({"id": dataset, "parameters": "Time," + ",".join(columns),
                       "time.min": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
                       "time.max": stop.strftime("%Y-%m-%dT%H:%M:%SZ"), "format": "json"})
    tag = start.strftime("%Y%m%dT%H%M%S") + "_" + stop.strftime("%Y%m%dT%H%M%S")
    cache_file = cache_directory / (dataset + "_" + tag + "_" + "-".join(columns) + ".json")

    if use_cache and not cache_file.exists():
        cache_file = cached_interval(dataset, "-".join(columns) + ".json", start, stop) or cache_file

    if use_cache and cache_file.exists():
        payload = json.loads(cache_file.read_text())
    else:
        url = "https://cdaweb.gsfc.nasa.gov/hapi/data?" + query
        with urlopen(url, timeout=120) as response:
            raw_response = response.read().decode("utf-8")
        try:
            payload = json.loads(raw_response)
        except json.JSONDecodeError as error:
            raise RuntimeError(f"NASA returned an invalid response: {raw_response[:200]}") from error
        if payload.get("status", {}).get("code") != 1200:
            raise RuntimeError(f"NASA HAPI request failed: {payload.get('status')}")
        if not payload.get("data"):
            raise ValueError(f"No data returned for {dataset}; check the dates.")
        cache_file.write_text(json.dumps(payload))

    frame = pd.DataFrame(payload["data"], columns=[p["name"] for p in payload["parameters"]])
    frame["Time"] = pd.to_datetime(frame["Time"], utc=True)
    frame = frame.set_index("Time").sort_index()
    for parameter in payload["parameters"][1:]:
        name = parameter["name"]
        frame[name] = pd.to_numeric(frame[name], errors="coerce")
        if parameter.get("fill") is not None:
            frame[name] = frame[name].replace(float(parameter["fill"]), np.nan)

    return frame.loc[(frame.index >= start) & (frame.index < stop)].rename(columns=columns)

In [ ]:
# HAPI parameters must follow the order in the dataset metadata.
solar_wind_columns = {"F": "Bmag", "BX_GSE": "Bx_GSE", "BY_GSM": "By_GSM", "BZ_GSM": "Bz_GSM",
                      "flow_speed": "speed", "proton_density": "density", "AE_INDEX": "AE", "AL_INDEX": "AL"}

def load_record(start, stop):
    """Download a record in monthly chunks, keeping gaps on a regular time axis."""
    chunk_edges = pd.DatetimeIndex([start, *pd.date_range(start, stop, freq="MS"), stop]).unique().sort_values()
    solar_wind_parts, dst_parts, thl_parts = [], [], []

    for left, right in zip(chunk_edges[:-1], chunk_edges[1:]):
        print(f"Loading {left:%Y-%m-%d} to {right:%Y-%m-%d}")
        solar_wind_parts.append(download_omni("OMNI_HRO_1MIN", solar_wind_columns, left, right, use_cache))
        dst_parts.append(download_omni("OMNI2_H0_MRG1HR", {"DST1800": "Dst"}, left, right, use_cache))
        tag = left.strftime("%Y%m%dT%H%M%S") + "_" + right.strftime("%Y%m%dT%H%M%S")
        thl_file = cache_directory / f"THL_{tag}_1min.csv"

        if use_cache and not thl_file.exists():
            thl_file = cached_interval("THL", "1min.csv", left, right) or thl_file

        if use_cache and thl_file.exists():
            part = pd.read_csv(thl_file, index_col="time")
            part.index = pd.to_datetime(part.index, utc=True)
        else:
            request = SwarmRequest()
            request.set_collection("SW_OPER_AUX_OBSM2_:THL")
            request.set_products(measurements=["B_NEC", "Quality"])
            part = request.get_between(left.to_pydatetime(), right.to_pydatetime()).as_dataframe(expand=True)
            part.index = pd.to_datetime(part.index, utc=True)
            part.index.name = "time"
            if part.empty:
                raise ValueError("THL download is empty; check coverage for the chosen dates.")
            part.to_csv(thl_file)
        thl_parts.append(part)

    minute_times = pd.date_range(start, stop, freq="1min", inclusive="left")
    solar_wind = pd.concat(solar_wind_parts).sort_index()
    solar_wind = solar_wind.loc[~solar_wind.index.duplicated()].reindex(minute_times)
    thl = pd.concat(thl_parts).sort_index()
    thl = thl.loc[~thl.index.duplicated()].reindex(minute_times)
    dst = pd.concat(dst_parts).sort_index()
    dst = dst.loc[~dst.index.duplicated()]
    if dst.empty:
        raise ValueError("No Dst data returned for this interval.")
    dst = dst.reindex(pd.date_range(start.floor("h") + pd.Timedelta("30min"), stop, freq="1h", inclusive="left"))

    return solar_wind, dst, thl

solar_wind, dst, thl = load_record(start, stop)
print("Missing fraction in THL components:")
print(thl[["B_NEC_N", "B_NEC_E", "B_NEC_C"]].isna().mean())
print("THL quality labels (D: definitive; Q: quasi-definitive):", thl["Quality"].value_counts().to_dict())

### Coordinates and a simple baseline

The code subtracts a constant background field, defined as the median over a selected reference interval. The resulting magnetic perturbations show changes relative to that interval, in nT.

The code also converts the measurements into magnetic east, north and up components using quasi-dipole (QD) coordinates, and calculates THL’s position in the model maps. These coordinates account for the geometry of Earth’s magnetic field. For details, (optional!) see [Laundal et al. (2016), equation 1](https://birkeland.uib.no/wp-content/uploads/2018/02/114-Laundal-JGR-OA.pdf).

Keep in mind that the reference interval affects the level of the plotted perturbations, and that the observed variations are not necessarily all caused by ionospheric currents.

In [ ]:
def prepare_thl(thl, start, stop, baseline_start, baseline_stop):
    """Subtract a constant baseline and calculate scaled QD components for this record."""
    thl = thl.copy()
    components = ["B_NEC_N", "B_NEC_E", "B_NEC_C"]
    reference = thl.loc[(thl.index >= baseline_start) & (thl.index < baseline_stop), components]
    if reference.notna().sum().min() < 60:
        raise ValueError("Choose a baseline interval with at least 60 valid samples per component.")
    baseline = reference.median()
    perturbation = thl[components] - baseline

    midpoint = start + (stop - start) / 2
    epoch = midpoint.year + (midpoint.dayofyear - 1) / 365.25
    dipole = Dipole(epoch=epoch)  # Used for model dipole tilt, not for component conversion.
    latitude  = float(thl["Latitude"].median())
    longitude = float(thl["Longitude"].median())
    radius_km = float(thl["Radius"].median()) / 1000

    # NEC north = -B_theta; NEC centre/down = -B_radial.
    geodetic_lat, height_km, delta_north_geo, delta_up_geo = geoc2geod(
        90 - latitude, radius_km, -perturbation["B_NEC_N"].to_numpy(), -perturbation["B_NEC_C"].to_numpy())
    delta_east_geo = perturbation["B_NEC_E"].to_numpy()
    apex = Apex(date=midpoint.to_pydatetime())
    qd_lat, qd_lon = apex.geo2qd(float(geodetic_lat), longitude, float(height_km))
    ma_lat, ma_lon = apex.geo2apex(float(geodetic_lat), longitude, 110.)
    f1, f2 = apex.basevectors_qd(float(geodetic_lat), longitude, float(height_km))
    F = f1[0] * f2[1] - f1[1] * f2[0]
    if F <= 0:
        raise ValueError("Unexpected QD base-vector orientation.")

    thl["dB_E_QD"] = (f1[0] * delta_east_geo + f1[1] * delta_north_geo) / F
    thl["dB_N_QD"] = (f2[0] * delta_east_geo + f2[1] * delta_north_geo) / F
    thl["dB_U_QD"] = delta_up_geo / np.sqrt(F)
    thl["MLT_QD"] = apex.mlon2mlt(np.full(len(thl), qd_lon), thl.index.tz_localize(None).to_pydatetime())

    print("Reference vector in geocentric NEC [nT]:", baseline.to_dict())
    print(f"THL: QD latitude {qd_lat:.2f}°, MA latitude at 110 km {ma_lat:.2f}°, QD scale factor F={F:.3f}")
    geometry = dict(apex=apex, dipole=dipole, midpoint=midpoint, qd_lat=qd_lat, qd_lon=qd_lon, ma_lat=ma_lat, ma_lon=ma_lon)
    return thl, geometry

thl, long_geometry = prepare_thl(thl, start, stop, baseline_start, baseline_stop)

### Inspect the longer record

Thin curves are hourly means and thick curves are daily means. Means require at least 75% data coverage. 

Briefly describe one pattern in the longer THL record that appears related to the IMF. Compare two intervals and give approximate values from the plots. Identify a relationship you would like to investigate during a shorter event in section 3. What would you expect to see if your interpretation is correct?

In [ ]:
def means_with_coverage(frame, frequency, expected):
    grouped = frame.resample(frequency)
    return grouped.mean().where(grouped.count() >= 0.75 * expected)

thl_hour = means_with_coverage(thl[["dB_N_QD", "dB_E_QD", "dB_U_QD"]], "1h", 60)
thl_day  = means_with_coverage(thl[["dB_N_QD", "dB_E_QD", "dB_U_QD"]], "1D", 1440)
sw_hour  = means_with_coverage(solar_wind, "1h", 60)
sw_day   = means_with_coverage(solar_wind, "1D", 1440)

fig, axes = plt.subplots(7, 1, figsize=(13, 14), sharex=True, constrained_layout=True)
for axis, component in zip(axes[:3], ["dB_N_QD", "dB_E_QD", "dB_U_QD"]):
    axis.plot(thl_hour.index, thl_hour[component], color="0.65", linewidth=0.6, label="Hourly mean")
    axis.plot(thl_day.index + pd.Timedelta("12h"), thl_day[component], color="black", label="Daily mean")
    axis.set_ylabel(component + "\n[nT]")
axes[0].legend(ncol=2)
for axis, component, color in zip(axes[3:5], ["By_GSM", "Bz_GSM"], ["tab:green", "tab:red"]):
    axis.plot(sw_hour.index, sw_hour[component], color=color, alpha=0.35, linewidth=0.7)
    axis.plot(sw_day.index + pd.Timedelta("12h"), sw_day[component], color=color)
    axis.set_ylim(-10, 10)
    axis.set_ylabel(component + "\n[nT]")
axes[5].plot(sw_hour.index, sw_hour["speed"], color="tab:blue")
axes[5].set_ylabel("Speed\n[km/s]")
axes[6].plot(dst.index, dst["Dst"], color="tab:purple")
axes[6].set_ylabel("Dst\n[nT]")
for axis in axes:
    axis.axhline(0, color="0.5", linewidth=0.5)
    axis.set_xlim(start, stop)
locator = mdates.AutoDateLocator()
axes[-1].xaxis.set_major_locator(locator)
axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator, tz=start.tz))
axes[-1].set_xlabel("Time [UTC]")
axes[0].set_title("THL magnetic perturbations and upstream conditions")
plt.show()

## 2. Compare southward and northward IMF

Run the supplied comparison for Bz = −5 and +5 nT, keeping By = 0, solar wind speed = 400 km/s, dipole tilt = 0 degrees, and F10.7 = 100. Compare the northern convection and electric potential maps. Describe the main difference in flow pattern and estimate a representative speed in each case using the arrow scale.

In one short paragraph or an annotated sketch, explain how the southward-IMF convection relates to the Dungey cycle: dayside reconnection, transport into the tail, nightside reconnection and return towards the dayside. Why does the sign of Bz matter? Refer to the northward-IMF map when discussing how the circulation changes.

Keep this section brief. Use the resulting physical picture in section 3. If the model code does not work, use the same inputs at [pyswipe.uib.no](https://pyswipe.uib.no/). You can still run the event data cells below.

In [ ]:
from pyswipe import SWIPE
from polplot import Polarplot
plt.rcParams["text.usetex"] = False  # Model imports can change Matplotlib defaults.

control_cases = []
for control_bz in [-5., 5.]:
    control_swipe = SWIPE(400., 0., control_bz, 0., 100., minlat=60., height=110.)
    vlat, vmlt = [coordinate.ravel() for coordinate in control_swipe.vectorgrid]
    slat, smlt = [coordinate.ravel() for coordinate in control_swipe.scalargrid]
    # Fixed reference mapping for reproducibility; this is not an event prediction.
    reference_date = pd.Timestamp("2020-03-20T12:00:00").to_pydatetime()
    reference_times = np.full(vlat.size, np.datetime64(reference_date))
    ve1, ve2 = control_swipe.get_convection_vel_MA(times=reference_times, apex_refdate=reference_date)
    potential = control_swipe.get_potential().ravel()
    north, north_scalar = vlat > 0, slat > 0
    control_cases.append({"Bz": control_bz, "vlat": vlat[north], "vmlt": vmlt[north],
                          "slat": slat[north_scalar], "smlt": smlt[north_scalar],
                          "potential": potential[north_scalar], "east": ve1[north], "poleward": -ve2[north]})

limit = max(1., max(np.nanmax(np.abs(case["potential"])) for case in control_cases))
levels = np.linspace(-limit, limit, 21)
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for axis, case in zip(axes, control_cases):
    polar = Polarplot(axis, minlat=60, plotgrid=False)
    polar.plotgrid(labels=False, color="0.6", linestyle=":")
    polar.writeLTlabels(size=12)
    polar.writeLATlabels(size=11)
    filled = polar.contourf(case["slat"], case["smlt"], case["potential"], levels=levels, cmap="RdBu_r", zorder=-2)
    polar.quiver(case["vlat"], case["vmlt"], case["poleward"], case["east"],
                 scale=5000, angles="xy", scale_units="xy", width=0.003,
                 qkeyproperties={"X": 0.15, "Y": 0.02, "U": 500, "label": "500 m/s"})
    axis.set_title(f"Hypothetical case: Bz = {case['Bz']:+g} nT")
fig.colorbar(filled, ax=list(axes), shrink=0.7, label="Potential [kV]")
plt.show()

## 3. Explain an event

Choose a 2–4 day interval containing an interesting change in solar wind or IMF conditions and a response at THL. You can choose an event outside May–June 2009 if you want, but it’s probably best to avoid a huge storm. Check that there are not major data gaps in the solar wind, THL and index data. Briefly explain your event choice.

### Choose the dates and download the data

The example below uses late June 2009. Change the event dates (!) and select two contrasting times. For another event, also change the baseline dates to a suitable reference interval near that event and review the assumed F10.7 value. 

The cells below reuse available data or download the new interval, and calculate QD components and THL positions using the event’s epoch. You do not need to change the dates in section 1. Rerun section 3 from here after changing dates.

In [ ]:
event_start = pd.Timestamp("2009-06-27T00:00:00Z")
event_stop  = pd.Timestamp("2009-07-01T00:00:00Z")
map_times   = [pd.Timestamp("2009-06-28T12:00:00Z"), pd.Timestamp("2009-06-29T12:00:00Z")]

# Reuse the long-record reference when possible; otherwise start with the first event day.
# Inspect this reference interval and change it if needed: the first day need not be quiet.
event_baseline_start = baseline_start if start <= event_start < event_stop <= stop else event_start
event_baseline_stop  = baseline_stop if start <= event_start < event_stop <= stop else event_start + pd.Timedelta("1D")
averaging_window     = pd.Timedelta("20min")
f107_assumed         = 70.0  # [sfu]; an assumption, not downloaded observations

if not event_start < event_stop or not event_baseline_start < event_baseline_stop:
    raise ValueError("Start times must precede stop times.")
if len(map_times) != 2 or len(set(map_times)) != 2 or any(not event_start <= time < event_stop for time in map_times):
    raise ValueError("Choose two different comparison times inside your event.")
if averaging_window <= pd.Timedelta(0):
    raise ValueError("Choose a positive averaging interval.")

In [ ]:
# Load the event and its averaging windows. Download the baseline separately if needed.
load_start = event_start - averaging_window
if start <= load_start and event_stop <= stop and use_cache:
    event_solar_wind = solar_wind.loc[load_start:event_stop].copy()
    event_dst       = dst.loc[load_start:event_stop].copy()
    event_raw_thl   = thl.loc[load_start:event_stop].copy()
else:
    event_solar_wind, event_dst, event_raw_thl = load_record(load_start, event_stop)

if start <= event_baseline_start < event_baseline_stop <= stop and use_cache:
    reference_thl = thl.loc[(thl.index >= event_baseline_start) & (thl.index < event_baseline_stop)]
elif load_start <= event_baseline_start < event_baseline_stop <= event_stop:
    reference_thl = event_raw_thl.loc[(event_raw_thl.index >= event_baseline_start) & (event_raw_thl.index < event_baseline_stop)]
else:
    _, _, reference_thl = load_record(event_baseline_start, event_baseline_stop)

combined_thl = pd.concat([event_raw_thl, reference_thl]).sort_index()
combined_thl = combined_thl.loc[~combined_thl.index.duplicated()]
converted_thl, event_geometry = prepare_thl(combined_thl, event_start, event_stop, event_baseline_start, event_baseline_stop)
thl_event = converted_thl.loc[(converted_thl.index >= event_start) & (converted_thl.index < event_stop)]
sw_event  = event_solar_wind.loc[(event_solar_wind.index >= event_start) & (event_solar_wind.index < event_stop)]
event_dst = event_dst.loc[(event_dst.index >= event_start) & (event_dst.index < event_stop)]
print("THL quality labels:", thl_event["Quality"].value_counts().to_dict())

In [ ]:
def coverage_row(series, cadence_minutes):
    missing = series.isna()
    groups = missing.ne(missing.shift()).cumsum()
    longest = missing.groupby(groups).sum().max() if len(series) else 0
    return {"available [%]": 100 * series.notna().mean(), "longest gap [min]": longest * cadence_minutes}

coverage = {}
for name in ["speed", "density", "By_GSM", "Bz_GSM", "AE"]:
    coverage[name] = coverage_row(sw_event[name], 1)
for name in ["dB_N_QD", "dB_E_QD", "dB_U_QD"]:
    coverage[name] = coverage_row(thl_event[name], 1)
coverage["Dst"] = coverage_row(event_dst["Dst"], 60)
print(pd.DataFrame(coverage).T.round(1).to_string())
print("Inspect the gaps in the plot below before settling on this event.")

### Inspect the event

This figure shows one-minute data and hourly Dst, with your two comparison times marked. Check the data coverage, including near those times. If a gap hides the change you want to explain, choose different times or another event.

In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(13, 14), sharex=True, constrained_layout=True)
axes[0].plot(sw_event.index, sw_event["speed"], color="tab:blue")
axes[0].set_ylabel("Speed\n[km/s]")
axes[1].plot(sw_event.index, sw_event["density"], color="tab:orange")
axes[1].set_ylabel("Density\n[cm$^{-3}$]")
axes[2].plot(sw_event.index, sw_event["By_GSM"], color="tab:green", label="By GSM")
axes[2].plot(sw_event.index, sw_event["Bz_GSM"], color="tab:red", label="Bz GSM")
axes[2].set_ylabel("IMF\n[nT]")
axes[2].legend(ncol=2)
for component, label in [("dB_N_QD", "North QD"), ("dB_E_QD", "East QD")]:
    axes[3].plot(thl_event.index, thl_event[component], label=label)
axes[3].set_ylabel("THL ΔB\n[nT]")
axes[3].legend(ncol=2)
axes[4].plot(thl_event.index, thl_event["dB_U_QD"], color="black")
axes[4].set_ylabel("THL ΔB up QD\n[nT]")
axes[5].step(event_dst.index, event_dst["Dst"], where="mid", color="tab:purple")
axes[5].set_ylabel("Dst\n[nT]")
axes[6].plot(sw_event.index, sw_event["AE"], color="black")
axes[6].set_ylabel("AE\n[nT]")
for axis in axes:
    axis.set_xlim(event_start, event_stop)
    for time in map_times:
        if event_start <= time < event_stop:
            axis.axvline(time, color="0.4", linestyle="--", linewidth=0.8)
locator = mdates.AutoDateLocator()
axes[-1].xaxis.set_major_locator(locator)
axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator, tz=event_start.tz))
axes[-1].set_xlabel("Time [UTC]")
axes[0].set_title("Event analysis: explaining THL from the solar wind and IMF")
# fig.savefig("THL_event_overview.png", dpi=200)
plt.show()

### Generate maps for the two times

Inputs to SWIPE/AMPS are trailing 20-minute averages of the solar wind data (with at least 75% coverage). Dipole tilt is calculated for each time. F10.7 is an assumed constant; state it in your report, or replace it with an appropriate observed value and cite its source. The input table below also gives the values to enter at [pyswipe.uib.no](https://pyswipe.uib.no/) if the Python model code does not work.

Both cases use common potential and arrow scales. The star marks THL (QD position for AMPS, modified-apex position at 110 km for SWIPE). Include the maps that help explain your observations. These are model predictions, not measured global patterns.

In [ ]:
input_rows = []
for time in map_times:
    if not event_start <= time < event_stop:
        raise ValueError("Each map time must lie inside your selected event.")
    window = event_solar_wind.loc[(event_solar_wind.index > time - averaging_window) & (event_solar_wind.index <= time), ["speed", "By_GSM", "Bz_GSM"]]
    expected = averaging_window / pd.Timedelta("1min")
    if window.notna().sum().min() < 0.75 * expected:
        raise ValueError(f"Insufficient solar wind data for {time}; choose another time or averaging window.")
    values = window.mean()
    tilt = float(np.asarray(event_geometry["dipole"].tilt([time.to_pydatetime()])).ravel()[0])
    input_rows.append({"time": time, "speed": values["speed"], "By": values["By_GSM"], "Bz": values["Bz_GSM"],
                       "tilt": tilt, "f107": f107_assumed})
model_inputs = pd.DataFrame(input_rows).set_index("time")
model_inputs["THL_MLT_QD"] = event_geometry["apex"].mlon2mlt(np.full(len(map_times), event_geometry["qd_lon"]), pd.DatetimeIndex(map_times).tz_localize(None).to_pydatetime())
model_inputs

In [ ]:
from pyswipe import SWIPE
from polplot import Polarplot
plt.rcParams["text.usetex"] = False  # Model imports can change Matplotlib defaults.

# Calculate first, so that every case can use the same plotting scales.
cases = []
for time, inputs in model_inputs.iterrows():
    swipe = SWIPE(inputs["speed"], inputs["By"], inputs["Bz"], inputs["tilt"], inputs["f107"], minlat=60., height=110.)
    vlat, vmlt = [coordinate.ravel() for coordinate in swipe.vectorgrid]
    slat, smlt = [coordinate.ravel() for coordinate in swipe.scalargrid]
    E_e, E_n = [np.asarray(value).ravel() for value in swipe.get_efield_MA(return_emphi_emlambda=True)]
    J_e, J_n = [np.asarray(value).ravel() for value in swipe.get_AMPS_current()]  # A/m; also initializes swipe.amps
    eq_e, eq_n = [np.asarray(value).ravel() / 1000 for value in swipe.amps.get_divergence_free_current(vlat, vmlt)]
    velocity_e1, velocity_e2 = [np.asarray(value).ravel() for value in swipe.get_convection_vel_MA(times=np.full(vlat.size, time.tz_localize(None).to_datetime64()), apex_refdate=event_geometry["midpoint"].tz_localize(None).to_pydatetime())]
    potential = swipe.get_potential().ravel()

    E_squared = E_e**2 + E_n**2
    hall_projection = np.divide(J_e * E_n - J_n * E_e, E_squared,
                                out=np.full_like(E_squared, np.nan), where=E_squared > 0.1**2)
    hall_e, hall_n = hall_projection * E_n, -hall_projection * E_e
    north, north_scalar = vlat > 0, slat > 0
    cases.append({"time": time, "mlat": vlat[north], "mlt": vmlt[north],
                  "slat": slat[north_scalar], "smlt": smlt[north_scalar], "potential": potential[north_scalar],
                  "eq_e": eq_e[north], "eq_n": eq_n[north], "hall_e": hall_e[north], "hall_n": hall_n[north],
                  "v_e": velocity_e1[north], "v_n": -velocity_e2[north]})

In [ ]:
potential_limit = max(np.nanmax(np.abs(case["potential"])) for case in cases)
potential_levels = np.linspace(-max(potential_limit, 1.), max(potential_limit, 1.), 19)
fig, axes = plt.subplots(len(cases), 3, figsize=(16, 6.2 * len(cases)), squeeze=False, constrained_layout=True)

for row, case in enumerate(cases):
    polar_axes = [Polarplot(axis, minlat=60, plotgrid=False) for axis in axes[row]]
    for polar in polar_axes:
        polar.plotgrid(labels=False, color="0.6", linestyle=":")
        polar.writeLTlabels(size=12)
    polar_axes[0].writeLATlabels(size=11)

    # Identical arrow scales in every row; both current panels use A/m.
    arrow_style = dict(angles="xy", scale_units="xy", width=0.003)
    for column, prefix in [(0, "eq"), (1, "hall"), (2, "v")]:
        scale, key, unit = (5000, 500, "m/s") if prefix == "v" else (2, 0.1, "A/m")
        polar_axes[column].quiver(case["mlat"], case["mlt"], case[prefix + "_n"], case[prefix + "_e"],
                                  scale=scale, color="black", **arrow_style,
                                  qkeyproperties={"X": 0.14, "Y": 0.02, "U": key, "label": f"{key} {unit}"})
        marker_lat, marker_lon = (event_geometry["qd_lat"], event_geometry["qd_lon"]) if column == 0 else (event_geometry["ma_lat"], event_geometry["ma_lon"])
        station_mlt = float(np.asarray(event_geometry["apex"].mlon2mlt(marker_lon, case["time"].to_pydatetime())))
        polar_axes[column].plot([marker_lat], [station_mlt], marker="*", color="magenta", markersize=12)

    filled = polar_axes[2].contourf(case["slat"], case["smlt"], case["potential"], levels=potential_levels,
                                   cmap="RdBu_r", extend="both", zorder=-2)
    fig.colorbar(filled, ax=axes[row, 2], shrink=0.7, label="Potential [kV]")
    for axis, title in zip(axes[row], ["AMPS divergence-free current", "Hall current", "Convection and potential"]):
        axis.set_title(title + f"\n{case['time']:%Y-%m-%d %H:%M} UTC")
plt.show()

### Supporting questions: connect the maps to THL

Use these prompts to build your explanation; they are not separate extended report questions.

- How do the convection and current directions near THL differ between your selected times? Which input changes could matter?
- Under a local infinite horizontal sheet approximation, the horizontal field **below** the sheet obeys B_north = μ0 J_east / 2 and J_east = −μ0 J_north / 2 in a local orthonormal east/north/up frame. Deduce what these directions are. 
- The Hall component is obtained by projecting the horizontal current perpendicular to E. How does it compare with the AMPS divergence-free current? Why might conductance structure and current closure matter?
- The AMPS divergence-free current is an equivalent-current-like quantity. It need not equal the local Hall current. What could one station tell you, and what would require spatial observations?


### Write your event explanation

Explain the links from solar wind conditions, through reconnection and magnetospheric circulation, to ionospheric convection and currents, and finally the magnetic field observed at THL. Support your text with the figures and approximate values:

- Identify the important solar wind changes, their approximate magnitudes and durations. Explain the significance of southward IMF, or its absence.
- Use your model maps to explain how convection and currents near THL differ between the selected times. Consider changes in both the driving inputs and THL’s magnetic local time.
- Explain how the current directions could produce the observed magnetic changes below the ionosphere.
- Which features resemble the relationship you identified in section 1, and which require a different or additional explanation? Consider whether differences in season or activity level matter.
- Discuss what Dst and AE add to your interpretation. Identify one feature that remains unexplained and what additional information would help.

Distinguish observations, model predictions and physical interpretation. You don’t need to explain every fluctuation (space is a mess). Focus on what looks like robust results supported by data. End with your main conclusion and its most important limitation.